# ISOM 835 · Session 10: Segments, Structure & Anomalies
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 23 · Prof. Hasan Arslan**

**Tonight in one sentence:** for nine weeks every dataset came with the answer column (*did the guest cancel?*). Tonight there is **no answer column at all**, and we let the data show us its own **groups**, its own **directions** and its own **oddballs**.

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it tells you **what the cell does** and **how to read the output**. You do not need to understand every line of code tonight. You do need to understand every *result*. Your numbers may differ from the slides in the last digit because library versions differ a little. The story does not change.

### Tonight's map

| Step | The question | Our tool | Its real name |
|---|---|---|---|
| **1–3** | "Who are our customers?" | turn 400,000 receipt lines into one row per customer | **RFM** features |
| **4–6** | "Why can't we just cluster?" | make days and pounds speak the same language | log + **standardizing** |
| **7** | "How does the computer find groups?" | the party with no name tags | **k-means** |
| **8** | "How many groups?" | how many T-shirt sizes to make | **elbow** and **silhouette** |
| **9–10** | "What do we do on Black Friday?" | name every group, meet one real customer in each | segment **profiles** |
| **11–12** | "What if the groups are not round?" | crowds and loners, a family tree, a maybe | **DBSCAN**, **hierarchical**, **Gaussian mixtures** |
| **13–14** | "Can I see 9 columns at once?" | the flashlight and the shadow | **PCA** (and UMAP) |
| **15** | "Which receipts don't belong?" | the game of 20 Questions | **Isolation Forest** |
| **16** | Your turn | four short questions | |

🎮 Tonight you play first: [**The Segments Explorer**](https://isom-835.vercel.app/explore/segments/) has five small games, one before each idea.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## The business situation: Black Friday is this Friday

You just joined the marketing team of a **UK online gift shop**. It sells all-occasion gifts: candle holders, lunch bags, Christmas decorations. Many of its customers are small shops that buy to resell. The data is **real**: every sale from **Dec 1, 2009 to Dec 9, 2010**.

Your boss says:

> *"Black Friday is this week. We can afford **four** different campaigns. Tell me who our customers are, which group gets which campaign, and while you're in there, tell me if anything in our sales data looks wrong."*

Notice what is **missing**. Nobody gives you a column called `segment`. There is **no right answer** to check against. Every model since Session 1 learned from an answer column (**supervised** learning). Tonight's models find structure **with no answers at all**. That is called **unsupervised learning**, and it is how most companies first make sense of their customers.

**Run the next cell.** It loads the tools and the data (about 7 MB, a few seconds).

In [ ]:
import warnings
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
warnings.filterwarnings('ignore')        # hide version notices from the libraries
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 38)

# One year of real sales from a UK online gift shop (UCI Online Retail II, CC BY 4.0), hosted in our course repo
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/online_retail_2010.csv.gz'
try:
    receipts = pd.read_csv(URL, parse_dates=['InvoiceDate'])
except Exception:                                         # backup copy on the class website
    receipts = pd.read_csv('https://isom-835.vercel.app/data/online_retail_2010.csv.gz', parse_dates=['InvoiceDate'])

print(f'{len(receipts):,} rows · {receipts["Invoice"].nunique():,} receipts · from {receipts["InvoiceDate"].min():%b %d, %Y} to {receipts["InvoiceDate"].max():%b %d, %Y}')
receipts.head(8)

## 1. Meet the data: one row is one line on a receipt

**How to read it:** each row is **one product on one receipt**. Invoice `489434` is one shopping basket: customer `13085` bought 12 Christmas glass balls at £6.95, 12 pink cherry lights at £6.75, and so on, on Dec 1, 2009 at 7:45 in the morning.

| column | means |
|---|---|
| `Invoice` | the receipt number. **Starts with C = a cancellation** (money going back) |
| `StockCode`, `Description` | the product |
| `Quantity`, `Price` | how many, and the price of one (in £) |
| `InvoiceDate` | when |
| `Customer ID` | who. **Sometimes empty** (a guest checkout) |
| `Country` | where the customer lives |

Our boss asked about **customers**, but the data is about **receipt lines**. The next three steps turn one into the other.

## 2. Clean up: two kinds of rows we cannot use

1. **Rows with no customer ID.** We cannot put a guest checkout into anyone's history.
2. **Cancellations** (invoice starts with `C`, quantity below zero). They are money going *back*, not a purchase.

The next cell counts both, removes them, and adds one column: `Amount` = quantity × price, the money on that line.

In [ ]:
no_id = receipts['Customer ID'].isna()
cancelled = receipts['Invoice'].astype(str).str.startswith('C')
print(f'rows with no customer ID: {no_id.sum():>9,}  ({no_id.mean():.0%} of all rows)')
print(f'cancellation rows:        {cancelled.sum():>9,}')

sales = receipts[~no_id & ~cancelled & (receipts['Quantity'] > 0) & (receipts['Price'] > 0)].copy()
sales['Customer ID'] = sales['Customer ID'].astype(int)
sales['Amount'] = sales['Quantity'] * sales['Price']
print(f'\nleft to work with: {len(sales):,} rows · {sales["Invoice"].nunique():,} receipts · {sales["Customer ID"].nunique():,} customers · £{sales["Amount"].sum():,.0f} in sales')

**Stop and notice:** about **1 row in 5** has no customer. That is normal in real data and a good thing to tell your boss: *"a fifth of our sales we cannot tie to a person."*

We are left with about **400,000 lines**, **19,000 receipts** and **4,312 customers**.

## 3. From receipts to customers: three numbers per person

Marketers have described customers with the same three numbers for decades. They are called **RFM**:

| letter | question | how we measure it |
|---|---|---|
| **R**ecency | *How long since they last bought?* | days from their last receipt to Dec 10, 2010 |
| **F**requency | *How often do they buy?* | number of different receipts |
| **M**onetary | *How much do they spend?* | total £ over the year |

Think of a coffee shop owner. She does not remember every latte you bought. She remembers **"was here yesterday, comes every morning, spends a lot."** That is R, F and M.

Watch it happen for one customer first. The next cell takes customer **12347** and squeezes all their receipt lines into three numbers.

In [ ]:
today = pd.Timestamp('2010-12-10')        # the day after the last sale in the data: "today" for our analysis

one = sales[sales['Customer ID'] == 12347]
print(f"customer 12347 has {len(one)} receipt lines on these receipts:")
print(one.groupby('Invoice').agg(date=('InvoiceDate', 'first'), lines=('StockCode', 'size'), spent=('Amount', 'sum')).round(2).to_string())

print('\nsqueezed into RFM:')
print(f"  Recency   = {(today - one['InvoiceDate'].max()).days} days since the last receipt")
print(f"  Frequency = {one['Invoice'].nunique()} receipts")
print(f"  Monetary  = £{one['Amount'].sum():,.2f}")

Now for **every** customer at once. In pandas this is one `groupby`: *for each customer, compute three summaries.* (You met `groupby` in Session 2.)

In [ ]:
rfm = sales.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda dates: (today - dates.max()).days),   # days since the last receipt
    Frequency=('Invoice', 'nunique'),                                     # number of different receipts
    Monetary=('Amount', 'sum'))                                           # total £ spent

print(f'{len(sales):,} receipt lines became {len(rfm):,} customers, one row each')
rfm.head()

**Quick check (30 seconds):** a customer has `Recency = 0`, `Frequency = 1`, `Monetary = 15`. Describe them in plain words. *(Answer: bought for the first time yesterday, and only spent £15. A brand-new customer.)*

## 4. Look before you cluster: the whales

Always look at the columns before you feed them to a model. The next cell prints the usual summary and draws a histogram of each column.

In [ ]:
print(rfm.describe().round(0).to_string())

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for j, col in enumerate(['Recency', 'Frequency', 'Monetary']):
    ax[j].hist(rfm[col], bins=50, color='#5b6bd8')
    ax[j].set_title(col)
ax[0].set_xlabel('days since last receipt'); ax[1].set_xlabel('receipts'); ax[2].set_xlabel('£ spent')
plt.tight_layout(); plt.show()

top5 = rfm['Monetary'].nlargest(5)
print(f"the 5 biggest customers spent £{top5.sum():,.0f} between them: {top5.sum() / rfm['Monetary'].sum():.0%} of all sales")

**How to read it:** the typical (median) customer bought **2 times** and spent about **£700**. But the biggest spent **£349,000**, and the 5 biggest customers alone bring in about **1 pound in 8**. Frequency and Monetary are piled up on the left with a very long tail to the right. These are **whales**: a few huge customers, mostly wholesalers.

Hold on to that word. It is about to cause trouble.

## 5. Why we can't just cluster: the scale problem

Clustering is about **distance**: customers that are *close* belong together. But close in what units? Recency is in **days** (0 to 373). Monetary is in **pounds** (3 to 349,000).

Two customers, A and B:
- they bought **300 days apart** (a huge difference in behaviour),
- and they spent **£300 apart** (a tiny difference).

To a computer both differences are just "300". Pounds swamp days, and the biggest spenders swamp everyone. It is like comparing people by height in **meters** and weight in **grams**: the weight number wins every argument.

Let's see what goes wrong. The next cell runs tonight's clustering tool (k-means, which we explain in Step 7) on the **raw** numbers and asks for 4 groups.

In [ ]:
raw_groups = KMeans(n_clusters=4, n_init=10, random_state=835).fit_predict(rfm[['Recency', 'Frequency', 'Monetary']])
print('customers in each of the 4 groups, RAW numbers:')
print(rfm.assign(group=raw_groups).groupby('group').agg(customers=('Recency', 'size'), avg_recency=('Recency', 'mean'),
      avg_frequency=('Frequency', 'mean'), avg_spend=('Monetary', 'mean')).round(0).sort_values('customers').to_string())

**Stop and notice:** almost **everyone (about 4,180 customers)** is in one giant group, and one "segment" has just **3 people**: the three biggest whales. The algorithm only looked at money. You cannot run a Black Friday campaign for 3 people and another one for "everybody else".

## 6. The fix: speak the same language

Two moves, both one line.

1. **Take the log** of Frequency and Monetary. The log squeezes the long tail: £100 → 4.6, £1,000 → 6.9, £100,000 → 11.5. Now each **×10** in spending is the same step, so a whale is "a big spender", not "a different planet". (We use `log1p`, the log of 1 + x, so a 0 stays safe.)
2. **Standardize** all three columns: subtract the average and divide by the spread. After that, every column is measured in the same unit: *"how unusual is this customer, compared to the typical one?"* `0` = typical, `+1` = one spread above, `-2` = two spreads below. (`StandardScaler`, the same tool as in Session 3.)

Think of currency exchange: before you compare prices from London, Tokyo and Boston, you convert them to **one currency**.

In [ ]:
rfm_log = pd.DataFrame({'Recency': rfm['Recency'],
                        'log Frequency': np.log1p(rfm['Frequency']),
                        'log Monetary': np.log1p(rfm['Monetary'])}, index=rfm.index)
Z = StandardScaler().fit_transform(rfm_log)          # every column: average 0, spread 1
Z_df = pd.DataFrame(Z, columns=['R', 'F', 'M'], index=rfm.index)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for j, col in enumerate(Z_df.columns):
    ax[j].hist(Z_df[col], bins=50, color='#1aa38a'); ax[j].set_title(f'{col} after log + standardize')
plt.tight_layout(); plt.show()
print(Z_df.describe().round(2).loc[['mean', 'std', 'min', 'max']].to_string())

**How to read it:** all three columns now have average **0** and spread **1**, and the long tails are tamed. The biggest whale is now about **+5** on M, not 170 times the typical customer. Now the three columns get **equal say**.

🎮 **Play it first:** [Station 1 of the Segments Explorer](https://isom-835.vercel.app/explore/segments/#s1) is a party with no name tags. Drop the hosts yourself, then let k-means do it.

## 7. How the computer finds groups: k-means

Imagine a big party where nobody wears a name tag. You want **4 conversation groups**. Here is a simple recipe:

1. **Place 4 hosts** somewhere in the room.
2. **Everyone walks to the nearest host.** That makes 4 groups.
3. **Each host moves to the middle of their group.**
4. **Repeat 2 and 3** until nobody switches groups.

That is the whole algorithm. The hosts are called **centroids** (centers), and *k* is the number of groups you ask for. Real name: **k-means**.

The next cell runs the recipe one round at a time on our customers, using two of the three columns (Recency and Frequency) so we can draw it on a flat page. Watch the diamonds (the hosts) move.

In [ ]:
XY = Z_df[['R', 'F']].values
rng = np.random.default_rng(1)
hosts = XY[rng.choice(len(XY), 4, replace=False)]        # step 1: 4 hosts at random customers
colors = np.array(['#1aa38a', '#5b6bd8', '#f5a524', '#d64545'])

fig, ax = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for rnd in range(4):
    nearest = np.argmin(((XY[:, None, :] - hosts[None]) ** 2).sum(-1), axis=1)      # step 2: everyone walks to the nearest host
    ax[rnd].scatter(XY[:, 0], XY[:, 1], c=colors[nearest], s=5, alpha=0.5)
    ax[rnd].scatter(hosts[:, 0], hosts[:, 1], c='black', marker='D', s=90)
    walking = ((XY - hosts[nearest]) ** 2).sum()
    ax[rnd].set_title(f'round {rnd + 1} · total walking {walking:,.0f}'); ax[rnd].set_xlabel('R (scaled): left = recent')
    hosts = np.array([XY[nearest == h].mean(axis=0) for h in range(4)])               # step 3: each host moves to the middle
ax[0].set_ylabel('F (scaled): up = frequent')
plt.tight_layout(); plt.show()

**How to read it:** each panel is one round. Dots are customers, colored by the host they walked to. The black diamonds are the hosts.

**Stop and notice:**
- the hosts start in poor spots and **move** toward the middle of their crowds each round;
- the **total walking** (the title) goes **down every round**. That number is what k-means makes as small as it can. Its real name is **inertia**: the total squared distance from every customer to their host;
- after a few rounds the hosts barely move. Done.

**Two traps, both fixed by one setting each:**
1. **Bad starting spots** can leave you stuck in a poor answer. The fix: start with spread-out hosts (**k-means++**, the default) and try several starts and keep the best (`n_init=10`).
2. **Unscaled columns**, which we fixed in Step 6.

Now the real thing, with all three columns, in one line:

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=835).fit(Z)       # 10 different starts, keeps the one with the least walking
rfm['segment'] = km.labels_
print(f'inertia (total walking): {km.inertia_:,.0f}')
print('customers per segment:', rfm['segment'].value_counts().sort_index().to_dict())

Four groups, and none of them is a group of 3. But is **4** the right number?

🎮 **Play it first:** [Station 2: How many T-shirt sizes?](https://isom-835.vercel.app/explore/segments/#s2)

## 8. How many segments? The T-shirt question

A clothing company measures thousands of bodies. How many sizes should it make?
- **1 size** fits nobody well.
- **20 sizes** fit everyone, but the factory and the shelves become a nightmare.
- **S, M, L, XL**: good enough fit, simple to run.

The number of sizes is a **business decision**, informed by the data. Choosing *k* is exactly the same. Two tools help:

1. **The elbow.** Total walking (inertia) **always** goes down when you add a group: more hosts, shorter walks. So "lowest inertia" always says "more groups". Look instead for the **bend**, where adding one more group stops helping much.
2. **The silhouette.** For each customer: *am I much closer to my own group than to the next group?* Like a wedding guest asking "do I belong at my table, or would I fit just as well at the next one?" Scores run from **-1** (wrong table) to **+1** (clearly the right table). Higher is better.

The next cell tries k = 2 to 8 (about 15 seconds).

In [ ]:
ks = range(2, 9)
inertia, silhouette = [], []
for k in ks:
    model = KMeans(n_clusters=k, n_init=10, random_state=835).fit(Z)
    inertia.append(model.inertia_)
    silhouette.append(silhouette_score(Z, model.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(list(ks), inertia, 'o-', color='#5b6bd8'); ax[0].set_title('The elbow: total walking (inertia)'); ax[0].set_xlabel('k = number of segments')
ax[1].plot(list(ks), silhouette, 'o-', color='#f5a524'); ax[1].set_title('Silhouette: do I belong at my table? (higher = better)'); ax[1].set_xlabel('k')
for a in ax: a.axvline(4, color='#d64545', ls='--', lw=1)
plt.tight_layout(); plt.show()
print(pd.DataFrame({'k': list(ks), 'inertia': np.round(inertia), 'silhouette': np.round(silhouette, 3)}).to_string(index=False))

**How to read it (be honest here):**
- The elbow bends around **k = 3 or 4**. After that each extra group saves less and less walking.
- The silhouette is highest at **k = 3** (about 0.41) and a little lower at **4** (about 0.37). After 4 it keeps falling.
- Rule of thumb: above **0.5** means clean, well-separated groups. **0.25 to 0.5** is what real business data usually looks like: real groups, but customers form more of a **continuum** than islands.

So the data says *"3 or 4"*. The boss says *"we can run four campaigns"*. **That settles it: k = 4.** Neither the elbow nor the silhouette can choose for you. They tell you which choices are reasonable, and the business picks among them.

🎮 Station 2 of the explorer shows the same thing.

## 9. Profile and name every segment

A group of customers with a number (segment 2) is not something marketing can act on. A segment needs a **profile** (what is typical about them), a **name**, and an **action**. The next cell computes the typical R, F and M of each segment **in the original units** (days, receipts, pounds), plus its size and its share of the money. Then it names the segments with a simple rule: the group that has been away longest is *Hibernating*, the biggest spenders are *Champions*, and of the other two the more frequent is *Loyal*.

In [ ]:
def profile_of(segments):
    p = rfm.assign(segment=segments).groupby('segment').agg(
        customers=('Recency', 'size'), days_since_last=('Recency', 'median'),
        receipts=('Frequency', 'median'), spend=('Monetary', 'median'), revenue=('Monetary', 'sum'))
    p['% of customers'] = (100 * p['customers'] / p['customers'].sum()).round(0)
    p['% of revenue'] = (100 * p['revenue'] / p['revenue'].sum()).round(0)
    return p.drop(columns='revenue')

def name_segments(p):
    names = {s: f'Segment {s}' for s in p.index}
    if len(p) == 4:
        hibernating = p['days_since_last'].idxmax(); champions = p['spend'].idxmax()
        rest = [s for s in p.index if s not in (hibernating, champions)]
        loyal = max(rest, key=lambda s: p.loc[s, 'receipts'])
        names.update({hibernating: 'Hibernating', champions: 'Champions', loyal: 'Loyal'})
        names.update({s: 'New & occasional' for s in rest if s != loyal})
    return names

profile = profile_of(rfm['segment'])
names = name_segments(profile)
rfm['name'] = rfm['segment'].map(names)
profile.index = profile.index.map(names)
order = ['Champions', 'Loyal', 'New & occasional', 'Hibernating']
profile = profile.reindex([n for n in order if n in profile.index])
print('typical customer in each segment (medians):')
profile

**How to read it** (medians: the typical member):

| segment | who they are | Black Friday action |
|---|---|---|
| **Champions** | bought less than 2 weeks ago, about 11 receipts this year, spent about £4,400. About **1 customer in 7, but about 2 pounds in 3** | **VIP early access** Thursday night. No discount needed: say thank you, keep them happy |
| **Loyal** | last bought about 5 weeks ago, 4 receipts this year, about £1,300 | **"Spend £X, get free shipping"**: grow the basket |
| **New & occasional** | bought in the last couple of months, but only once, about £330 | **Second-purchase coupon**: the hardest and most valuable step is the second visit |
| **Hibernating** | last seen about **8 months ago**, one purchase | **One cheap "we miss you" email**. A paid campaign costs more than they are worth |

**Stop and notice the money.** About **14%** of customers bring in about **64%** of revenue, and the Hibernating fifth brings in about **4%**. That one table is the Black Friday plan.

A segmentation is only as good as the four sentences that follow it. *That* is the deliverable, not the plot.

The next cell draws the four segments on the Recency × Frequency map.

In [ ]:
seg_colors = {'Champions': '#1aa38a', 'Loyal': '#5b6bd8', 'New & occasional': '#f5a524', 'Hibernating': '#9aa3b2'}
fig, ax = plt.subplots(figsize=(8, 4.6))
for n in profile.index:
    part = rfm[rfm['name'] == n]
    ax.scatter(part['Recency'], part['Frequency'], s=7, alpha=0.55, color=seg_colors.get(n, 'k'), label=f'{n} ({len(part):,})')
ax.set_yscale('log'); ax.set_xlabel('Recency: days since last receipt  (left = recent)'); ax.set_ylabel('Frequency: receipts (log scale)')
ax.set_title('Four segments, found with no labels'); ax.legend(markerscale=3); plt.show()

### Meet your four customers

"625 Champions" is still abstract. Marketing people think in **people**. The next cell picks, for each segment, the **most typical real customer**: the one closest to the segment's host (centroid). Then it reads their actual receipts.

In [ ]:
rfm['distance_to_host'] = ((Z - km.cluster_centers_[km.labels_]) ** 2).sum(axis=1)   # small = very typical of their segment
country = sales.groupby('Customer ID')['Country'].first()

for n in profile.index:
    cid = rfm.loc[rfm['name'] == n, 'distance_to_host'].idxmin()          # the most typical member
    mine = sales[sales['Customer ID'] == cid]
    top = mine.groupby(mine['Description'].str.strip().str.title())['Quantity'].sum().nlargest(3)
    print(f"━━ {n.upper()} · customer {cid} · {country[cid]}")
    print(f"   {mine['Invoice'].nunique()} receipt(s), {mine['InvoiceDate'].min():%b %d, %Y} → {mine['InvoiceDate'].max():%b %d, %Y}"
          f" · last seen {rfm.loc[cid, 'Recency']} days ago · £{rfm.loc[cid, 'Monetary']:,.0f} in total")
    print('   bought most: ' + ' · '.join(f'{q} × {d}' for d, q in top.items()) + '\n')

**Stop and notice:** now the segments are people.
- The typical **Champion** is a shop that restocks about **once a month**, all year. They bought **128 White Hanging Heart T-light holders**.
- The typical **Loyal** customer came back 4 times and bought **480 bubble-gum rings**.
- The typical **New & occasional** customer is a shop in **Spain** that tried us once in October: spinning tops and children's garden forks. Will they come back? That second visit is the Black Friday coupon's job.
- The typical **Hibernating** customer bought metal signs (*"Beware of the Cat"*) on **April 1, 2010**, and never came back.

When you present a segmentation, **show one real customer per segment**. People remember a person far better than a table of averages.

**Quick check:** your teammate says *"k-means found that there are exactly four types of customer."* What's wrong with that sentence? *(Answer: k-means found four groups because **we asked for four**. Ask for seven and it finds seven. It always finds groups, even in data that has none. The groups are useful, not "true".)*

## 10. Is it stable? Run it again

If a segmentation changes every time you press Run, you cannot build a campaign on it. The next cell reruns k-means with **5 different random starts** and measures how much the groups agree with ours (the **adjusted Rand index**: 1 = identical groups, 0 = no better than random).

In [ ]:
from sklearn.metrics import adjusted_rand_score
agree = [adjusted_rand_score(km.labels_, KMeans(n_clusters=4, n_init=10, random_state=seed).fit_predict(Z)) for seed in range(1, 6)]
print('agreement with our segments, 5 reruns:', '  '.join(f'{a:.2f}' for a in agree))

**Stop and notice:** about **0.97 to 1.00** every time. The segments hold. Customers near a border can switch, but the four groups and their names stay. Worth one line in your report: *"stable across reruns"*.

---

🎮 **Play it first:** [Station 3: Shapes k-means can't see](https://isom-835.vercel.app/explore/segments/#s3)

## 11. Other shapes: when the groups are not round

k-means has a hidden assumption: every group is a **round blob** around its host, and the blobs are about the **same size**. Real groups are not always like that. The next cell makes two crescent moons, an easy shape for your eye, and gives them to k-means and to a different method, **DBSCAN**.

**DBSCAN** thinks like someone looking down at a festival crowd: *wherever people are standing densely, that's a group; anyone standing alone is not part of any group.* Two dials replace *k*:
- `eps`: how close two people must be to count as neighbors;
- `min_samples`: how many neighbors make a spot "dense".

Points in no dense area get the label **-1: noise**. You never tell DBSCAN how many groups to find.

In [ ]:
from sklearn.datasets import make_moons
moons, _ = make_moons(n_samples=500, noise=0.06, random_state=835)
loners = np.random.default_rng(835).uniform([-1.4, -0.9], [2.4, 1.4], size=(15, 2))
shapes = np.vstack([moons, loners])

km_moons = KMeans(n_clusters=2, n_init=10, random_state=835).fit_predict(shapes)
db_moons = DBSCAN(eps=0.15, min_samples=6).fit_predict(shapes)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].scatter(shapes[:, 0], shapes[:, 1], c=np.array(['#5b6bd8', '#f5a524'])[km_moons], s=10); ax[0].set_title('k-means, k = 2: cuts straight across')
ax[1].scatter(shapes[:, 0], shapes[:, 1], c=['#cccccc' if l == -1 else ['#5b6bd8', '#f5a524', '#1aa38a', '#d64545'][l % 4] for l in db_moons], s=10)
ax[1].set_title(f'DBSCAN: {db_moons.max() + 1} groups found by itself + {(db_moons == -1).sum()} loners (gray)')
plt.tight_layout(); plt.show()

**Stop and notice:** k-means slices the moons with a straight cut, because it only knows "nearest host". DBSCAN follows the shape **and** sets the loners aside in gray. That gray "noise" pile is a free bonus: a list of points that don't belong anywhere.

Now DBSCAN on our customers (on the same scaled RFM numbers):

In [ ]:
db = DBSCAN(eps=0.3, min_samples=10).fit_predict(Z)
rfm['dbscan'] = db
print(f'DBSCAN found {db.max() + 1} dense groups by itself, and labeled {(db == -1).sum()} customers ({(db == -1).mean():.0%}) as noise\n')
print('the typical "noise" customer vs. everyone else (medians):')
print(rfm.assign(kind=np.where(db == -1, 'noise (loners)', 'in a dense group')).groupby('kind')[['Recency', 'Frequency', 'Monetary']].median().round(0).to_string())

**How to read it:** DBSCAN finds a handful of dense groups **without being told how many**, and sets aside a few hundred customers who sit in no crowd. Look who they are: the loners spend about **3 times more** and buy **twice as often** as everyone else, yet the typical one has **not been back for almost 5 months**. Big customers drifting away! DBSCAN's noise label is a quick way to find customers who need a **personal phone call**, not a mass email.

## 12. Two more ways to see groups

**Hierarchical clustering: a family tree.** Start with every customer alone. Join the two closest. Then the next two closest. Keep going until everyone is in one big family. The picture is called a **dendrogram**. You never choose *k* up front: you draw a horizontal line across the tree and count the branches you cut. Great for explaining to a manager *why four and not seven*.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
sample = Z_df.sample(300, random_state=835)
tree = linkage(sample, method='ward')            # 'ward' joins the pair that adds the least walking: k-means' cousin
plt.figure(figsize=(11, 3.6))
dendrogram(tree, no_labels=True, color_threshold=10.5)
plt.axhline(10.5, color='#d64545', ls='--'); plt.text(5, 11.2, 'cut here → 4 branches = 4 segments', color='#d64545')
plt.title('The family tree of 300 customers: cut it at any height'); plt.ylabel('how different the two joined groups are'); plt.show()

agg = AgglomerativeClustering(n_clusters=4, linkage='ward').fit_predict(Z)
print(f'family-tree segments vs. k-means segments, agreement: {adjusted_rand_score(km.labels_, agg):.2f}')

**How to read it:** the tall vertical lines are big differences between groups; short ones are near-twins. Cut high and you get a few broad segments; cut low and you get many small ones. The family tree's 4 segments agree with k-means' 4 fairly well (around **0.7**): the same broad story, with different borders. A good reminder that **borders between segments are a choice**.

**Gaussian mixtures: "maybe" is an answer.** k-means puts every customer in exactly one segment. But some customers really are in between. A **Gaussian mixture model** gives *probabilities*: this customer is 70% Loyal, 30% New. (We start it from the k-means hosts, so its four groups carry the same names.) The next cell finds the customers it is least sure about.

In [ ]:
gmm = GaussianMixture(n_components=4, means_init=km.cluster_centers_, random_state=835).fit(Z)   # start from the k-means hosts, so group g = segment g
membership = gmm.predict_proba(Z)                                 # one row per customer, 4 probabilities that add up to 1
unsure = membership.max(axis=1) < 0.7
print(f'{unsure.mean():.0%} of customers have no segment above 70%: they sit on a border')

i = np.where(unsure)[0][0]
print(f'\nexample: customer {rfm.index[i]} · R {rfm.Recency.iloc[i]} days, F {rfm.Frequency.iloc[i]} receipts, M £{rfm.Monetary.iloc[i]:,.0f}')
print('membership:', '  '.join(f'{names[g]}: {p:.0%}' for g, p in enumerate(membership[i]) if p >= 0.01))

**Stop and notice:**
- about **1 customer in 7** sits on a border. With k-means you would never know.
- Our old friend **customer 12347** (Step 3) bought 2 days ago, twice, and spent £1,323: about **half Champion, half New**. A future Champion, or a one-time big order? Honestly, we can't tell yet, and the model says so. For a customer like this you might test both offers, or start with the cheaper one.

---

🎮 **Play it first:** [Station 4: The flashlight](https://isom-835.vercel.app/explore/segments/#s4)

## 13. Seeing many columns at once: PCA

So far we used 3 columns and could draw two at a time. Real customer tables have many more. The next cell builds a richer profile: **9 columns** per customer (RFM, plus average basket in £, items per receipt, products per receipt, how many different products, how long they have been a customer, and how often they cancel).

Nobody can picture 9 dimensions. **PCA** (principal component analysis) makes the best possible **flat picture** of them.

**The flashlight analogy:** hold a coffee mug under a flashlight and look at its shadow on the wall. From one angle the shadow is a useless circle; from the side you see the handle and the shape. PCA turns the data (the mug) until the shadow shows **as much of the spread as possible**. The two best directions are called **PC1** and **PC2** (principal components).

In [ ]:
basket = sales.groupby(['Customer ID', 'Invoice']).agg(pounds=('Amount', 'sum'), items=('Quantity', 'sum'), lines=('StockCode', 'size'))
cancels = receipts[cancelled].groupby('Customer ID')['Invoice'].nunique()
customers = pd.DataFrame({
    'Recency': rfm['Recency'], 'Frequency': rfm['Frequency'], 'Monetary': rfm['Monetary'],
    'Basket £': basket.groupby('Customer ID')['pounds'].mean(),
    'Items/receipt': basket.groupby('Customer ID')['items'].mean(),
    'Products/receipt': basket.groupby('Customer ID')['lines'].mean(),
    'Different products': sales.groupby('Customer ID')['StockCode'].nunique(),
    'Days as customer': (today - sales.groupby('Customer ID')['InvoiceDate'].min()).dt.days,
    'Cancel rate': cancels.reindex(rfm.index).fillna(0) / rfm['Frequency']})

skewed = [c for c in customers.columns if c not in ('Recency', 'Days as customer')]
Z9 = StandardScaler().fit_transform(customers.assign(**{c: np.log1p(customers[c]) for c in skewed}))   # same recipe: log, then standardize
pca = PCA().fit(Z9)
signs = np.sign(pca.components_[:, [2, 1]].sum(axis=1)); signs[signs == 0] = 1   # pin the direction: + = more spend / more visits
pca.components_ *= signs[:, None]

kept = pca.explained_variance_ratio_
plt.figure(figsize=(7, 3)); plt.bar(range(1, 10), kept * 100, color='#5b6bd8'); plt.plot(range(1, 10), kept.cumsum() * 100, 'o-', color='#f5a524')
plt.xlabel('principal component'); plt.ylabel('% of the spread'); plt.title('How much each direction keeps (orange: running total)'); plt.show()
print(f'PC1 keeps {kept[0]:.0%}, PC2 keeps {kept[1]:.0%}: a flat picture with both keeps {kept[:2].sum():.0%} of the spread in 9 columns')

**How to read it:** the first direction alone keeps about **43%** of all the spread in 9 columns; the first two together keep about **61%**. A single flat picture shows well over half of what's going on. (This curve is called the **explained variance**.)

But what *is* PC1? Each component is a recipe: a weighted mix of the original columns. The weights are called **loadings**. Read them to give each axis a name.

In [ ]:
loadings = pd.DataFrame(pca.components_[:2].T, index=customers.columns, columns=['PC1', 'PC2'])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for j, pc in enumerate(['PC1', 'PC2']):
    ax[j].barh(loadings.index, loadings[pc], color=np.where(loadings[pc] > 0, '#1aa38a', '#d64545')); ax[j].set_title(f'{pc}: what it is made of'); ax[j].axvline(0, color='k', lw=0.8)
ax[0].invert_yaxis(); plt.tight_layout(); plt.show()
print(loadings.round(2).to_string())

**How to read it** (signs can flip on your screen; only the pattern matters):
- **PC1** is positive on almost everything: spend, frequency, products, basket size. Name: ***"how big a customer"***. Moving right on PC1 = a bigger customer in every way.
- **PC2** is positive on frequency and days-as-customer, negative on items and products per receipt. Name: ***"many small visits vs. a few big baskets"***: a regular who pops in often vs. a shop that orders in bulk now and then.

Now draw every customer on these two named axes, colored by the RFM segments from Step 9:

In [ ]:
P = pca.transform(Z9)[:, :2]
fig, ax = plt.subplots(figsize=(8, 5))
for n in profile.index:
    m = (rfm['name'] == n).values
    ax.scatter(P[m, 0], P[m, 1], s=6, alpha=0.5, color=seg_colors.get(n, 'k'), label=n)
ax.set_xlabel('PC1: how big a customer →'); ax.set_ylabel('PC2')
ax.text(0.01, 0.97, '↑ many small visits', transform=ax.transAxes, va='top', color='#555'); ax.text(0.01, 0.03, '↓ a few big baskets', transform=ax.transAxes, color='#555')
ax.set_title('4,312 customers, 9 columns, one flat picture'); ax.legend(markerscale=3, loc='lower right'); plt.show()

**Stop and notice:** the segments were found from **3** columns, yet they line up neatly along PC1 in a picture built from **9**. Champions on the right, Loyal in the middle, New and Hibernating on the left. The segmentation makes sense in the bigger picture too.

**Rule to keep:** *cluster on the scaled features, draw on the PCA picture.* PCA is for **seeing** and for squeezing many columns into a few. It is not a clustering method by itself.

## 14. Optional: UMAP, the subway map

PCA is a straight shadow. **UMAP** is more like a **subway map**: it keeps track of **who is next to whom** and bends everything else to make a clear picture. Stations that are neighbors stay neighbors, but the distances on the map are not real distances. Prettier pictures, less trustworthy distances. Never measure distances on a UMAP picture, and never cluster on one and call the groups real.

The next cell needs one extra library (about 30 seconds the first time). Skip it if you are short on time: nothing later depends on it.

In [ ]:
# OPTIONAL — UMAP picture (installs umap-learn, ~30 seconds)
!pip install -q umap-learn
import umap
U = umap.UMAP(n_neighbors=30, min_dist=0.2, random_state=835).fit_transform(Z9)
fig, ax = plt.subplots(figsize=(7, 5))
for n in profile.index:
    m = (rfm['name'] == n).values
    ax.scatter(U[m, 0], U[m, 1], s=5, alpha=0.5, color=seg_colors.get(n, 'k'), label=n)
ax.set_title('UMAP: neighbors kept, distances not'); ax.set_xticks([]); ax.set_yticks([]); ax.legend(markerscale=3); plt.show()

---

🎮 **Play it first:** [Station 5: 20 Questions](https://isom-835.vercel.app/explore/segments/#s5)

## 15. Anomalies: which receipts don't belong?

Your boss's last request: *"tell me if anything in our sales data looks wrong."* We have **19,000 receipts** and **no list** of which ones are wrong. Nobody can read them all.

**The 20 Questions analogy.** In the game, someone thinks of a thing and you ask yes/no questions until you know what it is. Now play it with receipts, asking **random** questions: *"Is the total above £400?" "Fewer than 3 products?"*
- An **ordinary** receipt looks like thousands of others. It takes **many** questions to single it out.
- A **strange** receipt is alone in its corner. It is cornered in just **a few** questions.

**Isolation Forest** plays this game hundreds of times with random questions (each game is a small random tree, Session 7) and gives every receipt a score: **the fewer questions it takes to isolate, the stranger it is.**

The next cell describes each receipt with 6 numbers (total £, items, different products, average price per item, highest price on it, hour of the day) and scores them all. No labels anywhere.

In [ ]:
receipt = sales.groupby('Invoice').agg(customer=('Customer ID', 'first'), country=('Country', 'first'), when=('InvoiceDate', 'first'),
                                       total=('Amount', 'sum'), items=('Quantity', 'sum'), products=('StockCode', 'nunique'),
                                       top_price=('Price', 'max'), first_line=('Description', 'first'))
receipt['avg_price'] = receipt['total'] / receipt['items']
receipt['hour'] = receipt['when'].dt.hour

features = np.log1p(receipt[['total', 'items', 'products', 'avg_price', 'top_price']]).assign(hour=receipt['hour'])
iso = IsolationForest(n_estimators=300, random_state=835).fit(features)
receipt['strangeness'] = -iso.score_samples(features)          # higher = isolated in fewer questions = stranger

print(f'{len(receipt):,} receipts scored. The 12 strangest:')
receipt.sort_values('strangeness', ascending=False).head(12)[['country', 'when', 'total', 'items', 'products', 'top_price', 'first_line', 'strangeness']].round(2)

**How to read it:** look at the `first_line` column. Most of the strangest receipts are a **single line called "Manual"**, often worth thousands of pounds. These are not gift sales at all. They are **bookkeeping entries** typed in by hand, hiding among the sales. Scroll a bit further (next cell) and you'll find entries like *"Adjustment by Peter on Jun 25 2010"*.

Nobody told the algorithm what "Manual" means. It simply noticed that these receipts look like nothing else.

But not everything strange is wrong. The next cell skips the bookkeeping lines and shows the strangest **real product** receipts.

In [ ]:
NOT_PRODUCTS = ['M', 'POST', 'C2', 'ADJUST', 'BANK CHARGES', 'PADS', 'TEST001', 'D', 'DOT', 'AMAZONFEE', 'S', 'B', 'CRUK']   # Manual, postage, adjustments, fees, tests
receipt['bookkeeping'] = sales.groupby('Invoice')['StockCode'].apply(lambda codes: codes.isin(NOT_PRODUCTS).any())
print(receipt[~receipt['bookkeeping']].sort_values('strangeness', ascending=False).head(8)[['country', 'total', 'items', 'top_price', 'first_line', 'strangeness']].round(2).to_string())

**Stop and notice:** vintage kitchen cabinets at **£295** each, in a shop where the typical item costs about £2. These are **real, honest sales**, just rare. Plus another adjustment by hand that was typed as a product.

**The big lesson:** *strange* is not the same as *wrong*. An anomaly score gives you a **short list for a human to look at**, not a verdict. (Just like the "not sure" pile in Session 9.)

### Did it really find the bookkeeping? Check with a hidden answer key

We actually *can* check this one. The cell above marked every receipt that contains a non-product line (Manual, postage, fees, adjustments). Isolation Forest **never saw** that column. Let's see how many of those it put at the top of its list.

In [ ]:
ranked = receipt.sort_values('strangeness', ascending=False)
print(f"in all receipts, {receipt['bookkeeping'].mean():.0%} contain a bookkeeping line\n")
for top in [25, 50, 100, 200]:
    share = ranked.head(top)['bookkeeping'].mean()
    print(f'  among the {top:>3} strangest: {share:4.0%}   ({share / receipt["bookkeeping"].mean():.0f}× more than by chance)')

**How to read it:** only **7%** of all receipts contain a bookkeeping line, but about **49 of the 50 strangest** do: about **14 times** more than chance. With **no labels**.

### The alert budget is a business decision

In a real company, someone has to **check** each flagged receipt. Suppose the finance team can check **50 receipts a week**. Then you send them your top 50. If they can check 200, you send 200 and accept that the list gets less pure (71% instead of 98%).

In scikit-learn, the dial for this is `contamination`: the share of rows you want flagged. `contamination=0.01` flags the strangest 1%.

**Where this is used:** card fraud (a stolen card is used unlike your usual purchases: see the Mastercard story on tonight's page), machines that are about to fail, network attacks, and, like tonight, **data errors** before they spoil your model. And notice the twist: those "Manual" entries are also inside our **Monetary** column from Step 3. Cleaning them out before segmenting is Your Turn Question 4.

## 16. Your turn

Four short questions. Each has a starter cell below.

1. **Five segments.** Rerun the profile with `n_clusters=5`. Which of tonight's four segments split in two? Would marketing treat the two halves differently on Black Friday?
2. **Country check.** Does the segmentation travel? Compare the share of each segment for **UK** customers vs. **customers outside the UK**. (Hint: each customer's country is in `sales`.)
3. **Alert budget.** The finance team can check **100 receipts a week**. Refit Isolation Forest with `contamination=100/len(receipt)`, count how many receipts it flags, and how many of them contain a bookkeeping line.
4. **Clean, then segment.** Remove the bookkeeping lines (`NOT_PRODUCTS`) from `sales`, rebuild RFM, and rerun k-means with k = 4. Do the segment sizes change much? Which segment changes most?

In [ ]:
# Your turn · Q1: five segments
km5 = KMeans(n_clusters=5, n_init=10, random_state=835).fit(Z)
profile_of(km5.labels_)

In [ ]:
# Your turn · Q2: does the segmentation travel? UK vs. the rest
country = sales.groupby('Customer ID')['Country'].first()
where = np.where(country.reindex(rfm.index) == 'United Kingdom', 'UK', 'outside UK')
pd.crosstab(where, rfm['name'], normalize='index').round(2)

In [ ]:
# Your turn · Q3: an alert budget of 100 receipts a week
iso100 = IsolationForest(n_estimators=300, contamination=100 / len(receipt), random_state=835).fit(features)
flagged = iso100.predict(features) == -1                 # -1 = "anomaly", 1 = "normal"
print(f'flagged: {flagged.sum()} receipts · with a bookkeeping line: {receipt.loc[flagged, "bookkeeping"].mean():.0%}')

In [ ]:
# Your turn · Q4: clean first, then segment
clean = sales[~sales['StockCode'].isin(NOT_PRODUCTS)]
rfm_clean = clean.groupby('Customer ID').agg(Recency=('InvoiceDate', lambda d: (today - d.max()).days),
                                             Frequency=('Invoice', 'nunique'), Monetary=('Amount', 'sum'))
Zc = StandardScaler().fit_transform(np.column_stack([rfm_clean['Recency'], np.log1p(rfm_clean['Frequency']), np.log1p(rfm_clean['Monetary'])]))
# ... your code: KMeans with k = 4 on Zc, then compare the segment sizes with tonight's

## What we learned tonight

1. **No answer column? Let the data show its structure.** That's unsupervised learning: groups, directions, oddballs. The business supplies the meaning.
2. **Turn rows into customers first.** One `groupby` turned 400,000 receipt lines into 4,312 RFM profiles.
3. **Scale before you cluster.** Unscaled, the whales took over (one "segment" of 3 people). Log the long tails, then standardize.
4. **k-means is a party with no name tags:** hosts, nearest, move to the middle, repeat. It always finds *k* groups, even when there are none.
5. **k is a T-shirt-size decision.** Elbow and silhouette say what's reasonable (3 or 4); the business decides (4 campaigns).
6. **A segment needs a name and an action.** About 14% of customers, the Champions, bring about 64% of revenue.
7. **Other shapes, other tools:** DBSCAN (crowds and loners), the family tree (hierarchical), "maybe" (Gaussian mixtures).
8. **PCA is the flashlight:** 9 columns, one flat picture, and the loadings name the axes. Cluster on the features, draw on the picture.
9. **Isolation Forest plays 20 Questions:** strange things are cornered fast. It found the bookkeeping entries hidden among the sales. Strange is not wrong: it's a short list for a human.

### Tonight's words and their real names

| what we said | the real name |
|---|---|
| no answer column | **unsupervised learning** |
| a host | **centroid** |
| total walking | **inertia** (within-cluster sum of squares) |
| "do I belong at my table?" | **silhouette score** |
| crowds and loners | **DBSCAN**: dense clusters and **noise** |
| the family tree | **dendrogram** (hierarchical / agglomerative clustering) |
| "half Champion, half New" | **soft assignment** (Gaussian mixture) |
| the shadow | **projection** onto **principal components** |
| what PC1 is made of | **loadings** |
| how much the shadow keeps | **explained variance** |
| 20 Questions | **Isolation Forest** (path length = anomaly score) |
| the alert budget | **contamination** |

### For your project
**Milestone 2** is due **tonight, Mon Nov 23, 11:59 PM**. **Milestone 3** (tuned model + error analysis + model card, from Session 9) is due **Mon Nov 30**. If your project has **no target column**, tonight's notebook is your template: features per unit, scale, choose *k* with a reason, profile and name, one action per segment.